# Stage B — Final Version (frozen MiniLM + logreg, trained on Sci-Sentence + FWS + SAL)

The best Stage-B head from the experiments: a **frozen `all-MiniLM-L6-v2` encoder + class-weighted
logistic regression**, trained on Sci-Sentence + FWS + SAL. A sentence is kept as a gap if a cue rule
fires **or** the head predicts limitation/future_work.

Reproduces the top result: gold end-to-end recall **0.571** (limitation 0.636, future_work 0.857).

## Setup

In [ ]:
# ===== Stage B helpers (self-contained; stdlib + sklearn + sentence-transformers) =====
import re, json
import numpy as np, pandas as pd
from pathlib import Path

# locate repo root + data
ROOT = Path.cwd()
while not (ROOT/"data"/"external").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT/"data"
EXT  = DATA/"external"
print("data dir:", EXT)

# ---- text / matching / cue rules (Stage B.1) ----
_W=re.compile(r"[a-z0-9]+"); _DEH=re.compile(r"(\w)-\s*(\w)")
_PH=re.compile(r"\{\{[^}]*\}\}"); _MA=re.compile(r"\\\(.*?\\\)|\\\[.*?\\\]|\$[^$]*\$"); _NW=re.compile(r"[^a-z0-9'\- ]+"); _WS=re.compile(r"\s+")
def norm(s): s=(s or "").lower();s=_PH.sub(" ",s);s=_MA.sub(" ",s);s=_NW.sub(" ",s);return _WS.sub(" ",s).strip()
def token_containment(n,h):
    nt=_W.findall(norm(_DEH.sub(r"\1\2",n)))
    if not nt: return 0.0
    hs=set(_W.findall(norm(_DEH.sub(r"\1\2",h)))); return sum(1 for t in nt if t in hs)/len(nt)
_FUT=re.compile(r"\b(future work|future direction|future research|future stud(?:y|ies)|in future|for future|as future|future investigation|leave[sd]?\s+(?:\w+\s+){0,6}?(?:for|to|as)\s+future|left\s+(?:\w+\s+){0,4}?future|remains?\s+(?:\w+\s+){0,4}?(?:future|open|interesting|valuable)|we\s+(?:plan|intend|aim|hope|would like|will)\s+to|we\s+(?:will|plan to|intend to)\s+explore|next step|further (?:work|research|investigation|study)|would be (?:interesting|valuable|worthwhile)|it would be interesting|an? (?:interesting|promising|natural) (?:direction|next step|avenue)|(?:plan|plans|planning)\s+to|intend(?:s|ed)?\s+to|also\s+intend|future directions? include|future plans|plans? include|we\s+will\s+(?:investigate|explore|extend|incorporate|study|consider|try|continue|apply|also)|will\s+(?:investigate|extend|incorporate|study|continue|consider)|to extend (?:our|the) (?:work|approach|method|model|framework|analysis|study)|another (?:interesting|promising|natural) (?:direction|avenue|line)|also (?:want|hope|interested|would like|plan|like) to|(?:promising|interesting|natural|fruitful) (?:future )?direction is|direction is to|investigate whether)\b",re.I)
_LIM=re.compile(r"\b(limitation|shortcoming|drawback|weakness|caveat|we did not|we do not|we have not|did not (?:study|consider|explore|address|evaluate|test)|is (?:restricted|limited|confined) to|are (?:restricted|limited|confined) to|only (?:considers?|applies?|works?|handles?)|remains? (?:unexplored|unaddressed|an open|unclear|unknown|challenging|elusive)|unexplored|unaddressed|beyond the scope|out of scope|assumes? (?:the (?:availability|existence)|access to)|(?:cannot|can not|fail[s]? to|do(?:es)? not (?:generalize|scale|extend))|not (?:sufficiently|fully|yet) (?:explored|studied|understood|addressed|clear)|lack[s]? (?:of|a )|adds? (?:latency|cost|overhead|complexity)|(?:suffers? from|prone to|sensitive to|struggles? with)|(?:formidable|profound|significant) (?:challenge|obstacle|difficulty))\b",re.I)
_CG=re.compile(r"\b(we (?:show|prove|establish|demonstrate|present|propose|introduce|derive)|in this (?:paper|work|section) we|our (?:main )?(?:result|theorem|contribution))\b",re.I)
def cue_label(s):
    f=_FUT.search(s); l=_LIM.search(s)
    if not f and not l: return None
    if f and not l: return "future_work"
    if l and not f:
        if _CG.search(s) and "limitation of our" not in s.lower():
            return "limitation" if re.search(r"\bour\b|\bthis (?:work|paper|approach|method|study)\b",s,re.I) else None
        return "limitation"
    return "future_work"
LABELS=["none","limitation","future_work"]

In [ ]:
# ---- dataset loaders -> rows {sentence,label,ds,group} ----
def load_sci(researchgap_as="future_work"):
    df=pd.read_excel(EXT/"scisentence_repo/datasets/Based_Version.xlsx")
    M={"LIMITATION":"limitation","EXTENSION":"future_work","RESEARCH GAP":researchgap_as,
       "OVERALL":"none","DESCRIPTION":"none","RESULT":"none","OTHER":"none"}
    return [{"sentence":str(r.Sentence),"label":M.get(str(r.Category).upper(),"none"),
             "ds":"sci","group":"sci::"+str(getattr(r,'Title',r.Id))[:40]} for r in df.itertuples()]
def load_fws(pos=2000,neg=4000):
    df=pd.read_csv(EXT/"fws_corpus.csv"); df["lab"]=df["label"].map({1:"future_work",0:"none"})
    p=df[df.lab=="future_work"].sample(min(pos,(df.lab=="future_work").sum()),random_state=0)
    n=df[df.lab=="none"].sample(min(neg,(df.lab=="none").sum()),random_state=0)
    return [{"sentence":str(r.text),"label":r.lab,"ds":"fws","group":"fws::"+str(r.id)} for r in pd.concat([p,n]).itertuples()]
def load_sal(neg=2000):
    df=pd.read_csv(EXT/"sal_rct/train.csv")
    df["lab"]=df["limitation_annotation"].astype(str).str.contains("True").map({True:"limitation",False:"none"})
    p=df[df.lab=="limitation"]; n=df[df.lab=="none"].sample(min(neg,(df.lab=="none").sum()),random_state=0)
    return [{"sentence":str(r.sentences),"label":r.lab,"ds":"sal","group":"sal::"+str(r.pmid)} for r in pd.concat([p,n]).itertuples()]

# gold eval (precomputed Stage-A candidates + gold gaps)
gold_eval=json.load(open(EXT/"gold_eval.json",encoding="utf-8"))
GS=[g["gap_sentence"] for g in gold_eval["gold"]]
def build_df(researchgap_as="future_work"):
    df=pd.DataFrame(load_sci(researchgap_as)+load_fws()+load_sal())
    df=df[df.sentence.str.len()>=20].reset_index(drop=True)
    df=df[~df.sentence.map(lambda s:max((token_containment(g,s) for g in GS),default=0)>=0.8)].reset_index(drop=True)
    return df

from sklearn.model_selection import GroupShuffleSplit
def split_per_dataset(df):
    tr=[]; val={}
    for ds in ["sci","fws","sal"]:
        sub=df[df.ds==ds].reset_index(drop=True)
        tri,vai=next(GroupShuffleSplit(1,test_size=0.15,random_state=0).split(sub,sub.label,sub.group))
        tr.append(sub.iloc[tri]); val[ds]=sub.iloc[vai].reset_index(drop=True)
    return pd.concat(tr).reset_index(drop=True), val

In [ ]:
# ---- frozen head + gold eval ----
import torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("device:",DEVICE)
_enc_cache={}
def get_encoder(name):
    if name not in _enc_cache: _enc_cache[name]=SentenceTransformer(name,device=DEVICE)
    return _enc_cache[name]
def embed(name,sents):
    if not len(sents): return np.zeros((0,384))
    return np.asarray(get_encoder(name).encode(list(sents),normalize_embeddings=True,batch_size=256,show_progress_bar=False))

cand=gold_eval["candidates"]; gold=pd.DataFrame(gold_eval["gold"])
def gold_recall(clf, encoder_name, use_rules=True):
    allc=[s for pid in cand for s in cand[pid]]; E=embed(encoder_name,allc)
    k=0; cemb={}
    for pid in cand: cemb[pid]=E[k:k+len(cand[pid])]; k+=len(cand[pid])
    kept=0; rec={"all":[0,0],"limitation":[0,0],"future_work":[0,0]}
    for pid in cand:
        hl=[list(clf.classes_)[int(r.argmax())] for r in clf.predict_proba(cemb[pid])] if len(cand[pid]) else []
        keep=[s for i,s in enumerate(cand[pid]) if (use_rules and cue_label(s)) or hl[i] in ("limitation","future_work")]
        kept+=len(keep); kt=" ".join(keep)
        for g in gold[gold.paper_id==pid].itertuples():
            hit=token_containment(g.gap_sentence,kt)>=0.7; rec["all"][1]+=1; rec["all"][0]+=int(hit)
            if g.gap_type in rec: rec[g.gap_type][1]+=1; rec[g.gap_type][0]+=int(hit)
    return {"kept":kept,**{f"recall_{k}":round(h/n,3) if n else None for k,(h,n) in rec.items()}}
def macro(y,p):
    present=[l for l in LABELS if l in set(y)]
    return round(f1_score(y,p,labels=present,average="macro",zero_division=0),3)

def train_frozen(encoder_name, tr, val, class_weight="balanced"):
    clf=LogisticRegression(C=2.0,max_iter=2000,class_weight=class_weight).fit(embed(encoder_name,tr.sentence.tolist()),tr.label.values)
    vm={ds:macro(val[ds].label.values,clf.predict(embed(encoder_name,val[ds].sentence.tolist()))) for ds in val}
    return clf, vm

## Train the final head

In [ ]:
FINAL_ENCODER="sentence-transformers/all-MiniLM-L6-v2"
df=build_df(); tr,val=split_per_dataset(df)
clf,vm=train_frozen(FINAL_ENCODER,tr,val)
print("trained on",len(tr),"sentences | per-dataset val macro-F1:",vm)

## The final Stage-B step

In [ ]:
def run_stage_b(sentences):
    """Keep a sentence if a cue rule fires OR the MiniLM+logreg head predicts a gap.
    Returns list of (sentence, gap_type, source)."""
    if not sentences: return []
    E=embed(FINAL_ENCODER,sentences)
    proba=clf.predict_proba(E); classes=list(clf.classes_)
    out=[]
    for i,s in enumerate(sentences):
        rule=cue_label(s); hl=classes[int(proba[i].argmax())]; hp=float(proba[i].max())
        if rule is not None: out.append((s,rule,"rule"))
        elif hl in ("limitation","future_work"): out.append((s,hl,f"head({hp:.2f})"))
    return out

# demo on one gold paper
pid=next(iter(gold_eval["candidates"])); cands=gold_eval["candidates"][pid]
kept=run_stage_b(cands)
print(f"paper {pid}: {len(cands)} candidates -> {len(kept)} kept\n")
for s,t,src in kept[:12]: print(f"[{t:12} {src:10}] {s[:140]}")

## Final gold metrics

In [ ]:
g_head=gold_recall(clf,FINAL_ENCODER,use_rules=False)
g_both=gold_recall(clf,FINAL_ENCODER,use_rules=True)
print("head only :",g_head)
print("rules+head:",g_both)

## Summary
`run_stage_b(sentences)` is the final classifier: cue rules give free high-precision accepts, the frozen
MiniLM + logreg head (trained on Sci-Sentence+FWS+SAL) catches the rest. Fed by Stage A, it is the cheap
replacement for sending whole papers to an LLM; output flows to Stage C. Gold recall ≈ 0.571.